# Third Eye - train the Sri Lankan rupee note detector

Runs MediaPipe Model Maker in Google Colab and produces `lkr_notes.tflite` for
`app/src/main/assets/`. Read `README.md` first for the photo and labelling protocol.

1. **Runtime > Change runtime type > T4 GPU**.
2. Zip your dataset folder as `lkr_dataset.zip` (layout below) and keep `train_lkr_notes.py` next to this notebook.
3. Run the cells top to bottom (about 20-60 minutes on a T4, depending on photo count).

```
lkr_dataset/
  train/       images/  Annotations/   (Pascal VOC)   or  images/ labels.json (COCO)
  validation/  ...
  test/        ...
```

In [ ]:
!pip install -q mediapipe-model-maker

Upload `train_lkr_notes.py` and `lkr_dataset.zip` (or mount Google Drive and copy them from there).

In [ ]:
from google.colab import files
uploaded = files.upload()   # pick train_lkr_notes.py and lkr_dataset.zip
!unzip -q -o lkr_dataset.zip -d /content
!ls /content/lkr_dataset

Train and export. Use `--format coco` if you exported COCO from Label Studio / Roboflow.
`--model multi_avg_384` is slower but better for small or several notes.

In [ ]:
!python train_lkr_notes.py --data /content/lkr_dataset --format voc --model multi_avg --epochs 60

The cell prints AP / AP50 / per-class AP (see README "Reading the evaluation") and downloads
`exported_model/lkr_notes.tflite`. Copy it to `app/src/main/assets/lkr_notes.tflite` and rebuild.

In [ ]:
# Quick sanity check: run the exported model on one test photo with the MediaPipe Tasks API.
!pip install -q mediapipe
import glob, mediapipe as mp
from mediapipe.tasks.python import vision, BaseOptions
det = vision.ObjectDetector.create_from_options(vision.ObjectDetectorOptions(
    base_options=BaseOptions(model_asset_path='exported_model/lkr_notes.tflite'),
    score_threshold=0.4, max_results=5))
photo = sorted(glob.glob('/content/lkr_dataset/test/images/*'))[0]
for d in det.detect(mp.Image.create_from_file(photo)).detections:
    c = d.categories[0]
    print(photo, '->', c.category_name, round(c.score, 2))